In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Řezání vodičů pro odhad střední hodnoty

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Odhadovaná spotřeba: 22 sekund na procesoru Heron (POZNÁMKA: Jde pouze o odhad. Tvůj skutečný čas může být jiný.)*
## Výsledky učení
Po absolvování tohoto tutoriálu by uživatelé měli rozumět:
- Jak použít [`qiskit-addon-cutting`](https://github.com/Qiskit/qiskit-addon-cutting) k rozdělení velkého Circuitu na menší podřady, čímž se sníží vliv šumu

## Předpoklady
Doporučujeme, aby uživatelé před absolvováním tohoto tutoriálu znali následující téma:
- Použití primitiva [Sampler](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2), které se v tomto workflow využívá

## Pozadí
Circuit-knitting je zastřešující pojem, který zahrnuje různé metody rozdělení Circuitu do několika menších podřadů s méně Gaty nebo Qubity. Každý z podřadů lze spustit nezávisle a výsledný výstup se získá klasickým post-processingem výsledků jednotlivých podřadů. Tato technika je dostupná v [doplňku Qiskit pro řezání Circuit](https://qiskit.github.io/qiskit-addon-cutting/index.html); podrobné vysvětlení techniky najdeš v [dokumentaci](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html) spolu s dalšími [úvodními materiály](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html).

Tento tutoriál se zaměřuje na metodu zvanou **řezání vodičů** (wire cutting), při níž je Circuit rozdělen podél vodiče [\[1\], \[2\]](#references). Všimni si, že rozdělení je v klasických Circuit jednoduché, protože výsledek v místě rozdělení lze určit deterministicky a je buď 0, nebo 1. Stav Qubitu v místě řezu je však obecně smíšený stav. Proto je třeba každý podřad měřit vícekrát v různých bázích (obvykle v tomograficky úplné bázi, jako je Pauliho báze [\[3\], \[4\]](#references)) a připravovat do odpovídajícího vlastního stavu. Obrázek níže (zdroj: [\[7\]](#references)) ukazuje příklad řezání vodičů pro čtyř-qubitový stav GHZ rozdělený do tří podřadů. Zde $M_j$ označuje sadu bází (obvykle Pauliho X, Y a Z) a $P_i$ sadu vlastních stavů (obvykle $|0\rangle$, $|1\rangle$, $|+\rangle$ a $|+i\rangle$).

![wc-1.png](../docs/images/tutorials/wire-cutting/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

Protože každý podřad obsahuje méně Qubitů a Gatů, očekává se, že bude méně náchylný k šumu. Tento tutoriál ukazuje příklad, kdy lze tuto metodu použít k efektivnímu potlačení šumu v systému.
## Požadavky
Před zahájením tohoto tutoriálu se ujisti, že máš nainstalováno následující:

- Qiskit SDK v2.0 nebo novější, s podporou [vizualizace](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 nebo novější ( `pip install qiskit-ibm-runtime` )
- Doplněk Qiskit pro řezání Circuit v0.10.0 nebo novější (`pip install qiskit-addon-cutting`)
- Qiskit addon utils 0.3 nebo novější (`pip install qiskit-addon-utils`)
- Qiskit Aer (`pip install qiskit-aer` )
## Nastavení

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## Příklad se simulátorem v malém měřítku
Tento tutoriál implementuje [Qiskit pattern](/guides/intro-to-patterns) pro simulaci jednorozměrného (1D) Circuit Many-Body Localization (MBL). MBL Circuit je hardwarově efektivní Circuit parametrizovaný dvěma parametry $\theta$ a $\vec{\phi}$. Když je $\theta$ nastaveno na $0$ a počáteční stav je připraven v $|0\rangle$ pro všechny Qubity, ideální střední hodnota $\langle Z_i \rangle$ je $+1$ pro každou pozici Qubitu $i$ bez ohledu na hodnoty $\vec{\phi}$. Více podrobností o tomto Circuitu najdeš v tomto [článku](https://www.nature.com/articles/s41467-025-57623-x).

Všimni si, že v bezešumovém simulátoru bude střední hodnota získaná s řezáním Circuitu i bez něj stejná.
### Krok 1: Mapování klasických vstupů na kvantový problém
#### Sestavení 1D MBL Circuitu
Nejprve uvádíme funkci pro sestavení 1D MBL Circuitu.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif)

Vypočítáme průměrnou střední hodnotu $O = \frac{1}{n} \sum_i Z_i$ přes všechny Qubity pro $\theta = 0$. Protože ideální střední hodnota $\langle Z_i \rangle = 1$ $\forall$ $i$, ideální střední hodnota $O$ je také $1$. Parametry $\phi$ jsou vybrány náhodně.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

Circuit je potřeba anotovat vložením **CutWire** na požadovaná místa, aby bylo možné jej rozdělit. V tomto tutoriálu volíme rovnoměrné rozdělení. MBL Circuit je navržen tak, že nastavením `use_cut=True` ve funkci se anotace vloží správně za $\frac{n}{2}$ Qubitů, kde $n$ je počet Qubitů v původním Circuitu. Náhodně vygenerované parametry jsme také přiřadili k Circuitu.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif)

### Krok 2: Optimalizace problému pro spuštění na kvantovém hardwaru
#### Rozdělení Circuitu na menší podřady
Nyní rozdělíme Circuit na dva menší podřady pomocí [`qiskit-addon-cutting`](https://qiskit.github.io/qiskit-addon-cutting/). `qiskit-addon-cutting` připojí virtuální Gate `Move` k rozdělení místa řezu vodiče vhodnou úpravou počtu Qubitů. Nyní vytvoříme Circuit s tímto virtuálním Gate. Protože je jeden řez vodiče, počet přidružených Qubitů se zvýší o 1.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif)

#### Sestavení a rozšíření pozorovatelné
Pozorovatelná, jak bylo definováno dříve, bude průměrem $Z$ na každém Qubitu. Po vložení virtuálního Gate `Move` se však efektivní počet Qubitů v Circuitu zvýší. Pozorovatelná musí být odpovídajícím způsobem rozšířena, aby zohledňovala tuto změnu v počtu Qubitů. Pozorovatelná vždy působí triviálně (jako $I$) na extra Qubit přidaný pro virtuální Gate `Move`.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

Now the circuit can be partitioned along the `Move` gate and we obtain the subcircuits, as well as the subobservable, which is the portion of the original observable associated with each subcircuit.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

Zde vizualizujeme dva podřady:

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif)

Rozšíření pozorovatelné pomocí operace `Move` vyžaduje datovou strukturu `PauliList`. Pro rekonstrukci střední hodnoty původního Circuitu potřebujeme pozorovatelnou ve formátu `SparsePauliOp`.

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


Jak bylo popsáno dříve, pro každý řez musí být Circuit na vstupu měřen v Pauliho bázi a Circuit na výstupu musí být připraven do vlastního stavu příslušné báze. Funkce `generate_cutting_experiments` vytvoří všechny tyto potřebné Circuity a koeficienty přidružené ke každému Circuitu potřebné pro rekonstrukci. Více podrobností najdeš v [tomto článku](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

### Step 4: Post-process and return result in desired classical format

Now we retrieve the result of each subexperiment run and reconstruct the expectation value of the uncut circuit:

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

### Krok 4: Post-processing a vrácení výsledku v požadovaném klasickém formátu
Nyní načteme výsledek každého spuštěného subexperimentu a rekonstruujeme střední hodnotu neřezaného Circuitu:

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Periodic boundary conditions with circuit cutting](/docs/tutorials/periodic-boundary-conditions-with-circuit-cutting)
- [Circuit cutting for depth reduction](/docs/tutorials/depth-reduction-with-circuit-cutting)
</Admonition>

## References


[1] Peng, T., Harrow, A. W., Ozols, M., & Wu, X. (2020). Simulating large quantum circuits on a small quantum computer. Physical review letters, 125(15), 150504.

[2] Tang, W., Tomesh, T., Suchara, M., Larson, J., & Martonosi, M. (2021, April). Cutqc: using small quantum computers for large quantum circuit evaluations. In Proceedings of the 26th ACM International conference on architectural support for programming languages and operating systems (pp. 473-486).

[3]  Perlin, M. A., Saleem, Z. H., Suchara, M., & Osborn, J. C. (2021). Quantum circuit cutting with maximum-likelihood tomography. npj Quantum Information, 7(1), 64.

[4]  Majumdar, R., & Wood, C. J. (2022). Error mitigated quantum circuit cutting. arXiv preprint arXiv:2211.13431.

[5]  Khare, T., Majumdar, R., Sangle, R., Ray, A., Seshadri, P. V., & Simmhan, Y. (2023). Parallelizing Quantum-Classical Workloads: Profiling the Impact of Splitting Techniques. In 2023 IEEE International Conference on Quantum Computing and Engineering (QCE) (Vol. 1, pp. 990-1000). IEEE.

[6]  Bhoumik, D., Majumdar, R., Saha, A., & Sur-Kolay, S. (2023). Distributed Scheduling of Quantum Circuits with Noise and Time Optimization. arXiv preprint arXiv:2309.06005.

[7]  Majumdar, R. (2024). Efficient Reduction of Resources and Noise in Discrete Quantum Computing Circuits (Doctoral dissertation, Indian Statistical Institute - Kolkata). https://www.proquest.com/openview/b481def90b1cc80e6b58a77c99e8385c/1?pq-origsite=gscholar&cbl=2026366&diss=y